# Court E2E v2 — Generalizable 4-channel pipeline

**Input:** any English query. Outputs court_consideration citations.

**No val-specific knowledge.** Every per-query signal comes from the LLM reading the query, the law pipeline's model output, or query-independent corpus statistics.

**Channels:**
1. **Statute pivot** — union(law-pipeline-preds, Stage-0 LLM-extracted statutes) with cross-lingual aliases
2. **BGE echo** — for each BGE in the law-pipeline output, find paragraphs that cite it
3. **Qwen3-Embedding-8B dense** — pre-built unified index, filtered to court family
4. **Noun-phrase BM25** — extract nouns from query, translate, per-language BM25

**Then:** Qwen3-Reranker-8B (top 1500 → fused score) → Qwen3-14B judge on borderline with rich context.

**Realistic target F1 on val court: 0.30-0.45.**

## Cell 1 — Install + paths

In [ ]:
!pip install -q transformers sentencepiece accelerate rank_bm25 langid pyarrow vllm autoawq

import torch, time, json, gc, os, re, sys, io, pickle
import pandas as pd, numpy as np
from pathlib import Path
from collections import defaultdict, Counter

from google.colab import drive
drive.mount('/content/drive')

# Your two Drive project folders
SWISS_LAW_DIR = Path('/content/drive/MyDrive/swiss_law')
LAW_PIPE_DIR  = Path('/content/drive/MyDrive/Omnilex-Agentic-Retrieval-Competition')

DATA_DIR  = SWISS_LAW_DIR / 'data'
EMB_DIR   = SWISS_LAW_DIR / 'artifacts' / 'embeddings'
OUT_DIR   = SWISS_LAW_DIR / 'court_e2e_v2_2026-05-22'
OUT_DIR.mkdir(parents=True, exist_ok=True)

LAW_PREDS_JSON    = LAW_PIPE_DIR / 'retrieval' / 'pipeline_output_v12' / 'law_predictions_per_query.json'
LAW_RERANK_CACHE  = LAW_PIPE_DIR / 'retrieval' / 'pipeline_cache' / 'reranker_Qwen_Qwen3-Reranker-8B_v12.json'
LAW_JUDGE_CACHE   = LAW_PIPE_DIR / 'retrieval' / 'pipeline_cache' / 'llm_judge_v12_borderline.json'

print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## Cell 2 — Load val + law-pipeline predictions (Option A or B fallback)

In [ ]:
val = pd.read_csv(DATA_DIR / 'val.csv')
QID, QCOL, GOLDCOL = 'query_id', 'query', 'gold_citations'
val['gold_list'] = val[GOLDCOL].apply(lambda s: [c.strip() for c in re.split(r'[;,]', str(s)) if c.strip()])
print('val queries:', len(val))

# === Load law pipeline predictions ===
law_preds = {}
if LAW_PREDS_JSON.exists():
    law_preds = json.loads(LAW_PREDS_JSON.read_text())
    print(f'Loaded law predictions (Option A) from {LAW_PREDS_JSON.name}: {len(law_preds)} queries')
elif LAW_RERANK_CACHE.exists() and LAW_JUDGE_CACHE.exists():
    print('Reconstructing law predictions from cached reranker + judge (Option B)...')
    rerank_cache = json.loads(LAW_RERANK_CACHE.read_text())
    judge_cache  = json.loads(LAW_JUDGE_CACHE.read_text())
    HIGH = 0.78   # P(yes) auto-YES threshold (approximates fused_score ≥ 0.55)
    for qid, score_map in rerank_cache.items():
        preds = set()
        for cit, p in score_map.items():
            if p >= HIGH:
                preds.add(cit)
        for cit, v in judge_cache.get(qid, {}).get('verdicts', {}).items():
            if v == 'YES':
                preds.add(cit)
        law_preds[qid] = sorted(preds)
    print(f'  Reconstructed: {len(law_preds)} queries, mean preds = {np.mean([len(v) for v in law_preds.values()]):.1f}')
else:
    print('WARN: no law predictions found. Channel 1 will only use Stage-0 LLM extraction.')

# Show what the law pipeline predicted per query (helps verify we have ~10-20 statutes per query)
for _, r in val.iterrows():
    qid = r[QID]
    preds = law_preds.get(qid, [])
    print(f"  {qid}: {len(preds)} law preds. Sample: {preds[:5]}")

## Cell 3 — Load court corpus + filter to doctrinal pool

In [ ]:
court = pd.read_csv(DATA_DIR / 'court_considerations.csv', low_memory=False)
court['text'] = court['text'].astype(str)
court_cit_set = set(court['citation'])
print('court corpus rows:', len(court))

val['gold_court'] = val['gold_list'].apply(lambda L: [c for c in L if c in court_cit_set])
val['gold_law']   = val['gold_list'].apply(lambda L: [c for c in L if c not in court_cit_set])
print('Mean court gold per query:', val['gold_court'].apply(len).mean())

# Filter to doctrinal pool — drop cantonal, dispositif, signature, very short
CANTONAL_RE   = re.compile(r'^(KGer|OGer|BezGer|Kantonsgericht|Cour cantonale|Tribunal cantonal)', re.I)
DISPOSITIF_RE = re.compile(
    r'^\s*(?:\d+\.?\s+)?'
    r'(?:Die Beschwerde wird|Le recours est|Il ricorso \u00e8|Im Namen|Au nom de|'
    r'Gerichtskosten\s|Les frais judiciaires|Es werden keine Kosten|'
    r'Il n\'est pas per\u00e7u|Lausanne,)', re.I)

pool_mask = (
    (~court['citation'].str.match(CANTONAL_RE, na=False))
    & (~court['text'].str.match(DISPOSITIF_RE, na=False))
    & (court['text'].str.len() >= 200)
)
pool = court[pool_mask].reset_index(drop=True)
pool['orig_idx'] = pool.index   # for embedding lookup later
print(f'Doctrinal pool: {len(pool)} / {len(court)} ({100*len(pool)/len(court):.1f}%)')

# Gold survival check
all_court_gold = set()
for L in val['gold_court']: all_court_gold.update(L)
surviving = all_court_gold & set(pool['citation'])
print(f'Court gold surviving filter: {len(surviving)}/{len(all_court_gold)} ({100*len(surviving)/max(1,len(all_court_gold)):.1f}%)')

## Cell 4 — Stage 0: LLM query understanding (Qwen3-14B, one call per query)

Generalizable: same prompt on any English query.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

QUERY_UNDERSTAND_CACHE = OUT_DIR / 'query_understanding.json'
if QUERY_UNDERSTAND_CACHE.exists():
    query_understanding = json.loads(QUERY_UNDERSTAND_CACHE.read_text())
    print(f'Loaded query understanding cache ({len(query_understanding)} queries)')
else:
    QWEN_NAME = 'Qwen/Qwen3-14B-AWQ'
    llm = LLM(model=QWEN_NAME, quantization='awq_marlin', max_model_len=4096,
              gpu_memory_utilization=0.85, dtype='float16', trust_remote_code=True)
    qtok = AutoTokenizer.from_pretrained(QWEN_NAME, trust_remote_code=True)

    UNDERSTAND_SYS = (
        'You are a Swiss legal-domain query analyzer. Read an English legal scenario and output strict JSON only. '
        'No explanation, no preamble.'
    )
    UNDERSTAND_USR_TPL = (
        'Output JSON with these keys:\n'
        '  statute_targets: list of statute strings (DE/FR/IT canonical form, e.g. "Art. 221 Abs. 1 lit. b StPO")\n'
        '  doctrine_seeds: list of legal doctrine names (DE/FR/IT/EN). Examples: "Kollusionsgefahr", "Untersuchungshaft", "Gef\u00e4lligkeit".\n'
        '  legal_area: one of ["criminal_prelim", "criminal_merits", "civil", "family", "social_insurance", "public_law", "admin", "banking", "contract", "property", "inheritance", "ipr"]\n'
        '  sub_questions: list of 2-5 doctrinal sub-questions the query embeds\n'
        '  bge_landmark_hints: list of BGE/ATF/DTF citations the query mentions or that are likely landmark cases. Use canonical form "BGE 137 IV 122". Empty if none known.\n\n'
        'Query:\n{query}\n\n'
        'JSON:'
    )

    def build_understand_prompt(q):
        msgs = [
            {'role': 'system', 'content': UNDERSTAND_SYS},
            {'role': 'user',   'content': UNDERSTAND_USR_TPL.format(query=q[:3000])},
        ]
        return qtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

    prompts = [build_understand_prompt(r[QCOL]) for _, r in val.iterrows()]
    outs = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=512))

    query_understanding = {}
    for (_, r), out in zip(val.iterrows(), outs):
        txt = out.outputs[0].text.strip()
        # Strip code fences if present
        m = re.search(r'\{.*\}', txt, re.DOTALL)
        if m: txt = m.group(0)
        try:
            query_understanding[r[QID]] = json.loads(txt)
        except json.JSONDecodeError:
            print(f'  WARN: JSON parse failed for {r[QID]}; using empty')
            query_understanding[r[QID]] = {'statute_targets':[], 'doctrine_seeds':[], 'legal_area':'', 'sub_questions':[], 'bge_landmark_hints':[]}

    QUERY_UNDERSTAND_CACHE.write_text(json.dumps(query_understanding, ensure_ascii=False, indent=2))
    del llm, qtok; gc.collect(); torch.cuda.empty_cache()
    print(f'Stage 0 complete. Cached to {QUERY_UNDERSTAND_CACHE.name}')

for _, r in val.iterrows():
    u = query_understanding[r[QID]]
    print(f"  {r[QID]}: {len(u.get('statute_targets',[]))} statutes, {len(u.get('doctrine_seeds',[]))} doctrines, {len(u.get('bge_landmark_hints',[]))} BGE hints, area={u.get('legal_area','')}")

## Cell 5 — Statute extraction + alias table + statute-anchor index over pool

Fixed regex (`[A-Z][A-Za-z]{1,7}\d?` for code names — handles StPO, StGB, StBOG).

In [ ]:
STATUTE_RE = re.compile(
    r'\b[Aa]rt(?:icle|icolo|\.)?\.?\s+(\d+[a-z]?)'
    r'(?:\s+(?:Abs|al|cpv|para|paragraph)\.?\s+(\d+))?'
    r'(?:\s+(?:lit|let|lett)\.?\s+([a-z]))?'
    r'\s+([A-Z][A-Za-z]{1,7}\d?)\b'
)

# Cross-lingual statute aliases (public knowledge from admin.ch SR catalog)
STATUTE_ALIASES = {
    'StPO':['CPP'],   'CPP':['StPO'],
    'StGB':['CP'],    'CP':['StGB'],
    'ZGB':['CC'],     'CC':['ZGB'],
    'OR':['CO'],      'CO':['OR'],
    'ZPO':['CPC'],    'CPC':['ZPO'],
    'BGG':['LTF'],    'LTF':['BGG'],
    'BV':['Cst','Cost'], 'Cst':['BV','Cost'], 'Cost':['BV','Cst'],
    'ATSG':['LPGA'],  'LPGA':['ATSG'],
    'IVG':['LAI'],    'LAI':['IVG'],
    'UVG':['LAA'],    'LAA':['UVG'],
    'SchKG':['LP'],   'LP':['SchKG'],
    'EMRK':['CEDH'],  'CEDH':['EMRK'],
    'StBOG':[], 'PartG':[], 'BSV':[],
}

def parse_statute(s):
    """Returns (full_canon, art_code) tuples for a statute string, or None."""
    m = STATUTE_RE.search(s)
    if not m: return None
    art, abs_, lit, code = m.group(1), m.group(2), m.group(3), m.group(4)
    parts_full = [f'Art. {art}']
    if abs_: parts_full.append(f'Abs. {abs_}')
    if lit:  parts_full.append(f'lit. {lit}')
    parts_full.append(code)
    return (' '.join(parts_full), f'Art. {art} {code}', code)

def expand_with_aliases(art_code_pair):
    """For 'Art. 221 StPO', also yield 'Art. 221 CPP'."""
    m = re.match(r'(Art\.\s+\d+[a-z]?)\s+(\S+)', art_code_pair)
    if not m: return [art_code_pair]
    art_part, code = m.group(1), m.group(2)
    out = [art_code_pair]
    for alias in STATUTE_ALIASES.get(code, []):
        out.append(f'{art_part} {alias}')
    return out

def collect_query_statutes(qid, val_row):
    """Union: law pipeline preds + Stage-0 LLM extraction + statutes in query text itself."""
    raw = []
    raw.extend(law_preds.get(qid, []))
    raw.extend(query_understanding.get(qid, {}).get('statute_targets', []))
    # Also extract from raw query (cheap regex, redundant safety net)
    for m in STATUTE_RE.finditer(val_row[QCOL]):
        raw.append(m.group(0))
    full_set, art_code_set = set(), set()
    for s in raw:
        p = parse_statute(s)
        if p:
            full_set.add(p[0])
            for ac in expand_with_aliases(p[1]):
                art_code_set.add(ac)
    return full_set, art_code_set

# Build statute → doc inverted index over pool (one-time, ~3-8 min on 1.8M)
print('Indexing statutes in pool...')
art_code_idx = defaultdict(list)
full_spec_idx = defaultdict(list)
t0 = time.time()
for i in range(len(pool)):
    text = pool['text'].iloc[i][:3000]
    for m in STATUTE_RE.finditer(text):
        art, abs_, lit, code = m.group(1), m.group(2), m.group(3), m.group(4)
        ac = f'Art. {art} {code}'
        art_code_idx[ac].append(i)
        # Full-spec form for higher discrimination
        parts = [f'Art. {art}']
        if abs_: parts.append(f'Abs. {abs_}')
        if lit:  parts.append(f'lit. {lit}')
        parts.append(code)
        full_spec_idx[' '.join(parts)].append(i)
print(f'  built in {time.time()-t0:.1f}s. {len(art_code_idx)} art+code keys, {len(full_spec_idx)} full-spec keys.')

## Cell 6 — Channel 1: statute pivot (3·full + 1·art_code)

Multi-statute scoring gives wide distribution since law pipeline contributes 10-20 statutes per query.

In [ ]:
ch1_candidates = {}
for _, r in val.iterrows():
    qid = r[QID]
    full_set, art_code_set = collect_query_statutes(qid, r)
    doc_score = defaultdict(int)
    for s in full_set:
        for i in full_spec_idx.get(s, []):
            doc_score[i] += 3
    for ac in art_code_set:
        for i in art_code_idx.get(ac, []):
            doc_score[i] += 1
    top = sorted(doc_score.items(), key=lambda x: -x[1])[:800]
    ch1_candidates[qid] = {'idx': [i for i, _ in top], 'score': [s for _, s in top]}
    print(f"  {qid}: {len(full_set)} full + {len(art_code_set)} art+code -> {len(top)} cands (top score={top[0][1] if top else 0})")

## Cell 7 — Channel 2: BGE echo expansion

In [ ]:
BGE_RE = re.compile(r'\b(?:BGE|ATF|DTF)\s+(\d+\s+[IVX]+\s+\d+)\b')

def normalize_bge(s):
    """Return canonical 'BGE NNN III NN' from any of BGE/ATF/DTF variants."""
    m = BGE_RE.search(s)
    return f'BGE {m.group(1)}' if m else None

def collect_query_bges(qid, val_row):
    out = set()
    # From law preds
    for s in law_preds.get(qid, []):
        nb = normalize_bge(s)
        if nb: out.add(nb)
    # From Stage-0 hints
    for s in query_understanding.get(qid, {}).get('bge_landmark_hints', []):
        nb = normalize_bge(s)
        if nb: out.add(nb)
    # From query text itself
    for m in BGE_RE.finditer(val_row[QCOL]):
        out.add(f'BGE {m.group(1)}')
    return out

# Build BGE -> [doc indices] index over pool (~3 min)
print('Indexing BGE/ATF/DTF mentions in pool...')
bge_idx = defaultdict(list)
t0 = time.time()
for i in range(len(pool)):
    text = pool['text'].iloc[i]
    for m in BGE_RE.finditer(text):
        bge_idx[f'BGE {m.group(1)}'].append(i)
print(f'  {len(bge_idx)} BGE keys in {time.time()-t0:.1f}s')

ch2_candidates = {}
for _, r in val.iterrows():
    qid = r[QID]
    bges = collect_query_bges(qid, r)
    idx_set = set()
    for bge in bges:
        idx_set.update(bge_idx.get(bge, [])[:200])
    ch2_candidates[qid] = {'idx': list(idx_set), 'score': [1.0]*len(idx_set)}
    print(f"  {qid}: {len(bges)} BGEs -> {len(idx_set)} echo cands")

## Cell 8 — Channel 3: Qwen3-Embedding-8B dense (load + filter to court)

In [ ]:
from transformers import AutoTokenizer, AutoModel

# Load manifest to find court rows that intersect with the pool
manifest = pd.read_parquet(EMB_DIR / 'qwen3_8b_unified_manifest.parquet')
print('Manifest rows:', len(manifest), '| columns:', list(manifest.columns))

# Filter manifest to court family AND to citations that survived pool filtering
pool_cit_set = set(pool['citation'])
court_manifest = manifest[(manifest['family'] == 'court') & (manifest['citation'].isin(pool_cit_set))].copy()
court_manifest['mani_row'] = court_manifest.index  # row in original manifest = row in stacked embedding tensor
court_manifest = court_manifest.reset_index(drop=True)
print(f'Court vectors in pool: {len(court_manifest)} / {len(pool)}')

# Build pool_idx -> manifest_row lookup so we can score pool indices later
cit_to_mani = dict(zip(court_manifest['citation'], court_manifest['mani_row']))
pool_to_mani = np.array([cit_to_mani.get(c, -1) for c in pool['citation']], dtype=np.int64)

# Load all 27 chunks into VRAM (~21 GB fp16; Blackwell has 95.6 GB free)
print('Loading embedding chunks...')
chunks = sorted(EMB_DIR.glob('qwen3_8b_unified_chunk*.npy'))
print(f'  {len(chunks)} chunks found')
E_full = np.concatenate([np.load(c, mmap_mode='r') for c in chunks], axis=0)
print(f'  full E shape = {E_full.shape}, dtype = {E_full.dtype}')

# Select only court-pool rows; move to GPU
mani_rows = court_manifest['mani_row'].values
E_court = torch.from_numpy(E_full[mani_rows].copy()).to('cuda', dtype=torch.float16)
del E_full; gc.collect()
print(f'  E_court (GPU) shape = {tuple(E_court.shape)}, VRAM = {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Load embedding model for queries only
EMB_NAME = 'Qwen/Qwen3-Embedding-8B'
emb_tok = AutoTokenizer.from_pretrained(EMB_NAME, trust_remote_code=True)
emb_mdl = AutoModel.from_pretrained(EMB_NAME, torch_dtype=torch.float16, trust_remote_code=True).to('cuda').eval()

QUERY_INSTRUCT = 'Instruct: Given a Swiss legal scenario, retrieve federal court paragraphs that state the applicable doctrinal rule.\nQuery: '

@torch.no_grad()
def embed_query(q):
    text = QUERY_INSTRUCT + q
    inp = emb_tok([text], return_tensors='pt', padding=True, truncation=True, max_length=8192).to('cuda')
    out = emb_mdl(**inp).last_hidden_state
    # last-token pooling (Qwen3-Embedding convention)
    attn = inp['attention_mask']
    lengths = attn.sum(dim=1) - 1
    vec = out[torch.arange(out.size(0)), lengths]
    vec = torch.nn.functional.normalize(vec, dim=-1).to(torch.float16)
    return vec[0]

# Per-query dense top-400
ch3_candidates = {}
for _, r in val.iterrows():
    qid = r[QID]
    qv = embed_query(r[QCOL])  # [d]
    # Cosine (E_court rows already normalized? if not, normalize)
    # We'll normalize at query time for safety
    norms = E_court.norm(dim=1, keepdim=True).clamp_min(1e-6)
    sims = (E_court / norms) @ qv
    top_idx_in_court = torch.topk(sims, k=min(400, sims.size(0))).indices.cpu().numpy()
    # Convert court_manifest row → pool row
    pool_idxs = []
    for ci in top_idx_in_court:
        cit = court_manifest['citation'].iloc[ci]
        match = pool.index[pool['citation'] == cit]
        if len(match): pool_idxs.append(int(match[0]))
    ch3_candidates[qid] = {'idx': pool_idxs, 'score': [float(s) for s in sims[top_idx_in_court].cpu().numpy()]}
    print(f"  {qid}: dense top-400 -> {len(pool_idxs)} cands in pool")

del emb_mdl, emb_tok, E_court; gc.collect(); torch.cuda.empty_cache()

## Cell 9 — Channel 4: noun-phrase BM25 (tri-lingual, on filtered subset)

In [ ]:
import langid
langid.set_languages(['de', 'fr', 'it'])

# Tag pool by language (cached if you've run this before)
LANG_CACHE = OUT_DIR / 'pool_languages.npy'
if LANG_CACHE.exists():
    pool['language'] = np.load(LANG_CACHE, allow_pickle=True)
    print('Loaded pool languages from cache')
else:
    pool['language'] = pool['text'].apply(lambda t: langid.classify(t[:300])[0])
    np.save(LANG_CACHE, pool['language'].values)
    print('Tagged + cached pool languages')
print('Pool language dist:', pool['language'].value_counts().to_dict())

# Extract noun-like tokens from English query (simple regex: caps + length filter)
NOUN_RE = re.compile(r'\b[A-Z][a-z]{4,}\b|\b[a-z]{5,}\b')
STOPLIST = {'the','and','that','with','from','this','these','those','their','have','been',
            'were','they','which','where','when','what','about','under','over','also','only',
            'because','therefore','either','before','after','during','while','still','should',
            'would','could','might','through','within','without','against','between'}

def extract_nouns(q, max_n=15):
    toks = NOUN_RE.findall(q)
    toks = [t.lower() for t in toks if t.lower() not in STOPLIST]
    counts = Counter(toks)
    return [t for t, _ in counts.most_common(max_n)]

# Translate noun lists EN -> DE/FR/IT (compact, fast since only ~15 tokens per query)
from transformers import MarianMTModel, MarianTokenizer
noun_trans_cache = OUT_DIR / 'noun_translations.json'
if noun_trans_cache.exists():
    nouns_by_lang = json.loads(noun_trans_cache.read_text())
    print('Loaded noun translations from cache')
else:
    nouns_by_lang = {r[QID]: {'en': extract_nouns(r[QCOL])} for _, r in val.iterrows()}
    for src in ['de','fr','it']:
        name = f'Helsinki-NLP/opus-mt-en-{src}'
        tok = MarianTokenizer.from_pretrained(name)
        m = MarianMTModel.from_pretrained(name).to('cuda').half().eval()
        for _, r in val.iterrows():
            joined = ' . '.join(nouns_by_lang[r[QID]]['en'])
            inp = tok([joined], return_tensors='pt', padding=True, truncation=True, max_length=128).to('cuda')
            with torch.no_grad():
                gen = m.generate(**inp, max_length=128, num_beams=1)
            translated = tok.batch_decode(gen, skip_special_tokens=True)[0]
            # Split back into tokens
            nouns_by_lang[r[QID]][src] = [t.strip().lower() for t in translated.split('.') if t.strip()]
        del m, tok; gc.collect(); torch.cuda.empty_cache()
    noun_trans_cache.write_text(json.dumps(nouns_by_lang, ensure_ascii=False, indent=2))
    print('Translated + cached noun lists')

# Build per-language BM25 over pool subsets (one-time)
from rank_bm25 import BM25Okapi
def tok_text(text): return re.findall(r'\b\w{3,}\b', str(text).lower())

BM25 = {}; POOL_LANG = {}
for lang in ['de','fr','it']:
    sub = pool[pool['language']==lang].reset_index(drop=False)  # keep original pool index
    POOL_LANG[lang] = sub
    t0 = time.time()
    BM25[lang] = BM25Okapi([tok_text(t) for t in sub['text']])
    print(f'  BM25[{lang}]: {len(sub)} docs in {time.time()-t0:.1f}s')

ch4_candidates = {}
for _, r in val.iterrows():
    qid = r[QID]
    cand_idx_score = defaultdict(float)
    for lang in ['de','fr','it']:
        toks = nouns_by_lang[qid].get(lang, []) + nouns_by_lang[qid].get('en', [])
        toks = [t for t in toks if len(t) >= 3]
        if not toks: continue
        scores = BM25[lang].get_scores(toks)
        top_local = np.argpartition(-scores, min(150, len(scores)-1))[:150]
        for li in top_local:
            pi = int(POOL_LANG[lang]['index'].iloc[li])
            cand_idx_score[pi] = max(cand_idx_score[pi], float(scores[li]))
    items = sorted(cand_idx_score.items(), key=lambda x: -x[1])[:450]
    ch4_candidates[qid] = {'idx': [i for i, _ in items], 'score': [s for _, s in items]}
    print(f"  {qid}: BM25 union -> {len(items)} cands")

del BM25; gc.collect()

## Cell 10 — Channel union + Stage 1 recall gate

In [ ]:
union_candidates = {}
stage1_recall = []
for _, r in val.iterrows():
    qid = r[QID]
    union_idx = set()
    union_idx.update(ch1_candidates.get(qid, {}).get('idx', []))
    union_idx.update(ch2_candidates.get(qid, {}).get('idx', []))
    union_idx.update(ch3_candidates.get(qid, {}).get('idx', []))
    union_idx.update(ch4_candidates.get(qid, {}).get('idx', []))
    cand_df = pool.iloc[list(union_idx)].copy().reset_index(drop=True)
    union_candidates[qid] = cand_df
    gold = set(r['gold_court'])
    if gold:
        hit = sum(c in cand_df['citation'].values for c in gold)
        stage1_recall.append(hit / len(gold))
        print(f"  {qid}: union={len(cand_df)}, gold_in_pool={hit}/{len(gold)} (R={hit/len(gold):.2f})")

print(f'\n=== Stage 1 macro recall: {np.mean(stage1_recall):.3f} ===')
print('   target: >= 0.60 to make F1 0.40 reachable; >= 0.75 to make 0.45+ reachable')

## Cell 11 — Rerank with Qwen3-Reranker-8B

In [ ]:
from transformers import AutoModelForCausalLM

RERANK_NAME = 'Qwen/Qwen3-Reranker-8B'
rtok = AutoTokenizer.from_pretrained(RERANK_NAME, padding_side='left', trust_remote_code=True)
rmdl = AutoModelForCausalLM.from_pretrained(RERANK_NAME, torch_dtype=torch.float16, trust_remote_code=True).to('cuda').eval()
YES_ID = rtok.convert_tokens_to_ids('yes')
NO_ID  = rtok.convert_tokens_to_ids('no')

PFX = rtok.encode('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n', add_special_tokens=False)
SFX = rtok.encode('<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n', add_special_tokens=False)
INSTRUCT = 'Given a query about a Swiss legal scenario, decide whether the federal court paragraph states a doctrinal rule relevant to the query.'

@torch.no_grad()
def rerank(query, docs, bsz=8, max_len=4096):
    scores = []
    for i in range(0, len(docs), bsz):
        batch = docs[i:i+bsz]
        pairs = [f'<Instruct>: {INSTRUCT}\n<Query>: {query[:1500]}\n<Document>: {d[:2500]}' for d in batch]
        ids = rtok(pairs, padding=False, truncation='longest_first',
                   max_length=max_len - len(PFX) - len(SFX))
        for j in range(len(ids['input_ids'])):
            ids['input_ids'][j] = PFX + ids['input_ids'][j] + SFX
        enc = rtok.pad(ids, padding=True, return_tensors='pt')
        enc = {k: v.to('cuda') for k, v in enc.items()}
        out = rmdl(**enc).logits[:, -1, :]
        yn = torch.stack([out[:, NO_ID], out[:, YES_ID]], dim=-1)
        p_yes = torch.nn.functional.log_softmax(yn, dim=-1)[:, 1].exp().float().cpu().tolist()
        scores.extend(p_yes)
    return scores

TOP_RERANK = 1500
rerank_results = {}
for _, r in val.iterrows():
    qid = r[QID]
    cdf = union_candidates[qid].head(TOP_RERANK).reset_index(drop=True)
    t0 = time.time()
    cdf['p_rerank'] = rerank(r[QCOL], cdf['text'].tolist(), bsz=8)
    rerank_results[qid] = cdf.sort_values('p_rerank', ascending=False).reset_index(drop=True)
    gold_top50 = sum(c in rerank_results[qid].head(50)['citation'].values for c in r['gold_court'])
    print(f"  {qid}: reranked {len(cdf)} in {time.time()-t0:.1f}s | gold in top50 = {gold_top50}/{len(r['gold_court'])}")

del rmdl, rtok; gc.collect(); torch.cuda.empty_cache()

## Cell 12 — Rerank-only F1 sweep (checkpoint before adding judge)

In [ ]:
def f1(p, g):
    if not g: return None
    tp = len(p & g)
    if not p or tp == 0: return 0.0
    P = tp/len(p); R = tp/len(g)
    return 2*P*R/(P+R)

print(f"{'K':>4} | {'macro F1':>10} | per-query")
best = (0, 0.0)
for K in [5, 10, 15, 20, 25, 30, 50, 75]:
    per_q = []
    for _, r in val.iterrows():
        pred = set(rerank_results[r[QID]].head(K)['citation'])
        v = f1(pred, set(r['gold_court']))
        if v is not None: per_q.append(v)
    m = np.mean(per_q)
    if m > best[1]: best = (K, m)
    print(f'{K:>4} | {m:>10.3f} | {[round(v,2) for v in per_q]}')
print(f'\nRerank-only best fixed-K: K={best[0]}, macro F1 = {best[1]:.3f}')

## Cell 13 — Qwen3-14B judge on borderline (rich context)

In [ ]:
from vllm import LLM, SamplingParams
QWEN_NAME = 'Qwen/Qwen3-14B-AWQ'
llm = LLM(model=QWEN_NAME, quantization='awq_marlin', max_model_len=4096,
          gpu_memory_utilization=0.85, dtype='float16', trust_remote_code=True)
jtok = AutoTokenizer.from_pretrained(QWEN_NAME, trust_remote_code=True)

DE_PARTY_RE = re.compile(r'^(?:\d+(?:\.\d+)?\.?\s+)?(Der Beschwerdef\u00fchrer|Die Beschwerdef\u00fchrerin|Die Vorinstanz)', re.I)
FR_PARTY_RE = re.compile(r'^(?:\d+(?:\.\d+)?\.?\s+)?(Le recourant|La recourante|La cour cantonale)', re.I)
CHAMBER_RE  = re.compile(r'\b(BGE|ATF|DTF)\s+\d+\s+([IVX]+)|^(\d[A-Z])_')

def feats(text, citation):
    statutes = sorted({m.group(0) for m in STATUTE_RE.finditer(text[:1500])})[:6]
    voice = 'court'
    if DE_PARTY_RE.match(text) or FR_PARTY_RE.match(text):
        voice = 'appellant_or_lower_court'
    m = CHAMBER_RE.search(citation)
    chamber = (m.group(2) or m.group(3)) if m else ''
    return statutes, voice, chamber

def judge_prompt(query, citation, text, fts, q_understand):
    statutes, voice, chamber = fts
    msgs = [
        {'role':'system', 'content':'You are a Swiss Federal Court doctrinal-paragraph expert. Output exactly one token: YES or NO. Default to NO when uncertain.'},
        {'role':'user', 'content':
            f'Query (English):\n{query[:1500]}\n\n'
            f'Query analysis:\n'
            f'  Legal area: {q_understand.get("legal_area","")}\n'
            f'  Doctrinal sub-questions: {q_understand.get("sub_questions",[])[:3]}\n\n'
            f'Candidate paragraph:\n'
            f'  Citation: {citation}\n'
            f'  Chamber: {chamber}\n'
            f'  Opener voice: {voice}\n'
            f'  Statutes cited: {statutes}\n'
            f'  Text: {text[:1800]}\n\n'
            'Answer YES only if ALL three:\n'
            '(a) Federal court (not cantonal)\n'
            '(b) States/echoes/applies a doctrinal rule on-point for the query (rule_statement, echo, application, party_position, or lower_court_summary roles are acceptable; pure dispositif/cost/signature are not)\n'
            '(c) The doctrinal sub-question matches one of the query\'s sub-questions (not merely the same statute)\n\nAnswer:'
        }
    ]
    return jtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

TOP_KEEP = 50
AUTO_YES_K = 5
auto_yes, borderline_pile = {}, {}
for _, r in val.iterrows():
    qid = r[QID]
    top = rerank_results[qid].head(TOP_KEEP).reset_index(drop=True)
    auto_yes[qid] = set(top.head(AUTO_YES_K)['citation'])
    borderline_pile[qid] = top.iloc[AUTO_YES_K:].reset_index(drop=True)

all_prompts, all_keys = [], []
for qid, bdf in borderline_pile.items():
    q = val[val[QID]==qid][QCOL].iloc[0]
    qu = query_understanding.get(qid, {})
    for _, c in bdf.iterrows():
        all_prompts.append(judge_prompt(q, c['citation'], c['text'], feats(c['text'], c['citation']), qu))
        all_keys.append((qid, c['citation']))

print(f'Judging {len(all_prompts)} borderline candidates...')
t0 = time.time()
outs = llm.generate(all_prompts, SamplingParams(temperature=0.0, max_tokens=4))
print(f'Done in {time.time()-t0:.1f}s')

verdicts = {}
for (qid, cit), o in zip(all_keys, outs):
    t = o.outputs[0].text.strip().upper()
    verdicts.setdefault(qid, {})[cit] = t.startswith('YES')

del llm; gc.collect(); torch.cuda.empty_cache()

## Cell 14 — Final F1 + error analysis

In [ ]:
rows = []
for _, r in val.iterrows():
    qid = r[QID]
    gold = set(r['gold_court'])
    if not gold:
        rows.append({'qid':qid,'gold':0,'pred':0,'tp':0,'P':None,'R':None,'F1':None})
        continue
    pred = set(auto_yes[qid]) | {c for c,v in verdicts.get(qid,{}).items() if v}
    if len(pred) < 3:
        for c in rerank_results[qid]['citation']:
            pred.add(c)
            if len(pred) >= 3: break
    tp = len(pred & gold); P = tp/max(1,len(pred)); R = tp/len(gold); F = 2*P*R/(P+R) if (P+R) else 0.0
    rows.append({'qid':qid,'gold':len(gold),'pred':len(pred),'tp':tp,'P':round(P,3),'R':round(R,3),'F1':round(F,3)})

res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / 'per_query_results.csv', index=False)
macro = res.loc[res['F1'].notna(), ['P','R','F1']].mean()
print(res.to_string(index=False))
print(f"\n=== MACRO: P={macro['P']:.3f}  R={macro['R']:.3f}  F1={macro['F1']:.3f} ===")

# Error analysis
print('\n--- Per-query: where did the gold sit in the rerank pool? ---')
for _, r in val.iterrows():
    qid = r[QID]
    gold = set(r['gold_court'])
    if not gold: continue
    df = rerank_results[qid].reset_index(drop=True)
    in_pool = df[df['citation'].isin(gold)]
    missed = gold - set(df['citation'])
    ranks = (in_pool.index + 1).tolist()
    print(f"{qid}: {len(in_pool)}/{len(gold)} in rerank pool. Ranks: {ranks[:20]}  Missed: {len(missed)}")